# TVM Prep Guide

This is the main guide for the repository. The markdown files explain how to open the devcontainer and install dependencies; this notebook explains how to use the project once the container is ready.

Run this notebook from top to bottom after these setup commands have completed in the repository root:

```bash
bash .devcontainer/scripts/install-python-deps.sh requirements.txt
bash .devcontainer/scripts/build-tvm.sh
```

The notebook starts with a tiny offline PyTorch model so the first pass is fast and deterministic. After that, it explains how the same helpers are used for larger models, other frontends, target profiles, Python runtime validation, and the C++ runner.

## What TVM Is Doing Here

Apache TVM is a compiler stack for machine-learning models. In this repository the workflow has five stages:

1. Start from a model in a framework such as PyTorch, TensorFlow/Keras, ONNX, or TFLite.
2. Convert that model into TVM Relay, which is TVM's high-level graph representation.
3. Compile Relay for a target profile, such as local x86_64 Linux or Raspberry Pi AArch64.
4. Export graph-executor artifacts: graph JSON, parameter bytes, compiled library, and metadata.
5. Load those artifacts in a runtime and run inference.

The notebook uses the same helper modules as the command-line tools under `examples/python/`, so the small example and the real model workflow exercise the same code path.

In [7]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

EXAMPLES = ROOT / 'examples' / 'python'
if str(EXAMPLES) not in sys.path:
    sys.path.insert(0, str(EXAMPLES))

print('Python:', sys.version.split()[0])
print('Repository:', ROOT)
print('Examples package:', EXAMPLES)

Python: 3.11.15
Repository: /workspaces/TVM-Prep-guide
Examples package: /workspaces/TVM-Prep-guide/examples/python


## Validate The Environment

A TVM setup can fail in several places: the Python frameworks may be missing, TVM may not find its compiled shared libraries, or the cross-compilers may not exist. This cell checks those pieces before any model compilation begins.

The imports run in subprocesses so noisy framework startup logs, especially TensorFlow CUDA messages, do not make the notebook look failed when the import actually succeeded.

In [8]:
import shutil
import subprocess

from tvm_prep.targets import TARGET_PROFILES

checks = {
    'TVM': 'import tvm; print(tvm.__version__)',
    'PyTorch': 'import torch; print(torch.__version__)',
    'TorchVision': 'import torchvision; print(torchvision.__version__)',
    'TensorFlow': 'import tensorflow as tf; print(tf.__version__)',
    'ONNX': 'import onnx; print(onnx.__version__)',
    'ONNX Runtime': 'import onnxruntime; print(onnxruntime.__version__)',
}

for label, code in checks.items():
    result = subprocess.run([sys.executable, '-c', code], text=True, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(f'{label} import failed:\n{result.stderr}')
    print(f'{label}:', result.stdout.strip().splitlines()[-1])
print('\nToolchain:')
for tool in ['gcc', 'g++', 'aarch64-linux-gnu-gcc', 'arm-linux-gnueabihf-gcc', 'emcc', 'cmake', 'ninja']:
    print(f'{tool}:', shutil.which(tool))

TVM: 0.19.0
PyTorch: 2.3.1+cu121
TorchVision: 0.18.1+cu121
TensorFlow: 2.16.2
ONNX: 1.16.2
ONNX Runtime: 1.18.1

Toolchain:
gcc: /usr/bin/gcc
g++: /usr/bin/g++
aarch64-linux-gnu-gcc: /usr/bin/aarch64-linux-gnu-gcc
arm-linux-gnueabihf-gcc: /usr/bin/arm-linux-gnueabihf-gcc
emcc: /usr/bin/emcc
cmake: /usr/bin/cmake
ninja: /usr/bin/ninja


## Pick A Target Profile

A TVM target describes the machine code TVM should generate. The same model can be compiled for local Linux, Raspberry Pi, WebAssembly, Vulkan, or C source export by changing the profile.

For this notebook, use `x86_64` because it runs inside the devcontainer. Cross targets are useful after local validation works.

In [9]:
for name, profile in sorted(TARGET_PROFILES.items()):
    print(f'{name}')
    print(f'  target: {profile.target}')
    print(f'  host:   {profile.host or profile.target}')
    print(f'  output: model.{profile.output_format}')
    if profile.cc:
        print(f'  cc:     {profile.cc}')
    print(f'  notes:  {profile.notes}\n')

TARGET_PROFILE = 'x86_64'

c
  target: c
  host:   c
  output: model.tar
  notes:  C source export path used as the bridge toward microTVM-style deployment.

native
  target: llvm -mcpu=native
  host:   llvm -mcpu=native
  output: model.so
  cc:     gcc
  notes:  Best for local validation only; do not use for portable artifacts.

raspi4_aarch64
  target: llvm -mtriple=aarch64-linux-gnu -mcpu=cortex-a72 -mattr=+neon
  host:   llvm -mtriple=aarch64-linux-gnu
  output: model.so
  cc:     aarch64-linux-gnu-gcc
  notes:  Raspberry Pi 4 running a 64-bit Linux OS.

raspi_armv7
  target: llvm -mtriple=armv7-linux-gnueabihf -mcpu=cortex-a72 -mattr=+neon,+vfp3
  host:   llvm -mtriple=armv7-linux-gnueabihf
  output: model.so
  cc:     arm-linux-gnueabihf-gcc
  notes:  32-bit Raspberry Pi Linux target.

vulkan
  target: vulkan
  host:   llvm -mtriple=x86_64-linux-gnu
  output: model.so
  notes:  GPU target for Vulkan-capable hosts. Use instead of CUDA in this guide.

wasm32
  target: llvm -mtriple=wasm32-unknown-unknown-was

## Compile A Small Model

The first compile uses a tiny in-notebook PyTorch model so the notebook does not need network access or downloaded model weights. A pretrained ResNet compile is useful, but it requires downloading weights and takes longer.

This small classifier still exercises the important TVM steps: tracing a PyTorch module, converting it to Relay, compiling it, exporting artifacts, and running those artifacts.

`LoadedModel` is the repository's simple contract between model loading and compilation. It records the frontend, input name, input shape, dtype, and optional labels.

In [10]:
import torch

from tvm_prep.model_zoo import LoadedModel


class TinyImageClassifier(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.features = torch.nn.Sequential(
            torch.nn.Conv2d(3, 8, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = torch.nn.Linear(8, 4)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


torch.manual_seed(0)
torch.set_grad_enabled(False)

loaded = LoadedModel(
    name='tiny_image_classifier',
    frontend='pytorch',
    model=TinyImageClassifier().eval(),
    input_name='input0',
    input_shape=(1, 3, 32, 32),
)

print(loaded)

LoadedModel(name='tiny_image_classifier', frontend='pytorch', model=TinyImageClassifier(
  (features): Sequential(
    (0): Conv2d(3, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): AdaptiveAvgPool2d(output_size=(1, 1))
  )
  (classifier): Linear(in_features=8, out_features=4, bias=True)
), input_name='input0', input_shape=(1, 3, 32, 32), input_dtype='float32', labels=None)


## Import The Model Into Relay

Relay is TVM's graph IR. For PyTorch, the helper creates a random example tensor with the declared input shape, traces the module with TorchScript, and calls `relay.frontend.from_pytorch`. The output is an `IRModule` plus parameter tensors.

At this stage TVM has understood the model graph, but it has not generated target-specific machine code yet.

In [11]:
from tvm_prep.compile import import_to_relay

mod, params = import_to_relay(loaded)
print(type(mod).__name__)
print('parameter tensors:', len(params))
print('Relay import succeeded for:', loaded.name)

IRModule
parameter tensors: 4
Relay import succeeded for: tiny_image_classifier


## Compile And Export Artifacts

`build_and_export` compiles the Relay module with `relay.build` and writes a portable artifact directory:

- `model.json`: graph-executor graph
- `model.params`: serialized parameter dictionary
- `model.so`, `model.tar`, or `model.wasm`: compiled target library
- `metadata.json`: model name, input name, input shape, target profile, and library filename

Generated outputs go under `examples/artifacts/`, which is ignored by git except for `.gitkeep`.

In [13]:
import json

from tvm_prep.compile import build_and_export
from tvm_prep.targets import get_target_profile

profile = get_target_profile(TARGET_PROFILE)

artifact_dir = build_and_export(loaded, profile, ROOT / 'examples' / 'artifacts')
print('Artifacts:', artifact_dir.relative_to(ROOT))
for path in sorted(artifact_dir.iterdir()):
    print('-', path.name)

metadata = json.loads((artifact_dir / 'metadata.json').read_text())
metadata

Artifacts: examples/artifacts/tiny_image_classifier/x86_64
- metadata.json
- model.json
- model.params
- model.so


{'model': 'tiny_image_classifier',
 'frontend': 'pytorch',
 'input_name': 'input0',
 'input_shape': [1, 3, 32, 32],
 'input_dtype': 'float32',
 'target_profile': 'x86_64',
 'target': 'llvm -mtriple=x86_64-linux-gnu -mcpu=x86-64',
 'host': None,
 'library': 'model.so'}

## Run The Exported Artifacts With Python

The Python runtime path mirrors what a deployment program does:

1. Read `metadata.json` to learn the input name and compiled library file.
2. Load `model.json`, `model.params`, and the compiled library.
3. Create a TVM graph executor module.
4. Set the model input, run inference, and read output tensor 0.

The input below is deterministic synthetic data. For ImageNet models, the CLI uses image preprocessing from `tvm_prep.preprocess` instead.

In [14]:
import numpy as np

from tvm_prep.runtime import run_graph_executor, topk

input_data = np.linspace(-1.0, 1.0, num=np.prod(loaded.input_shape), dtype='float32').reshape(loaded.input_shape)
output = run_graph_executor(artifact_dir, input_data)

print('output shape:', output.shape)
for rank, (idx, prob, label) in enumerate(topk(output, k=4), start=1):
    print(f'{rank}: id={idx} p={prob:.6f} {label}')

output shape: (1, 4)
1: id=1 p=0.393344 <no-label>
2: id=2 p=0.261390 <no-label>
3: id=0 p=0.183968 <no-label>
4: id=3 p=0.161298 <no-label>


## Compile Real Models With The CLI

Once the small model works, use the CLI for real models. The CLI calls the same helper functions used above, but adds argument parsing for frontends, model files, target profiles, and output directories.

PyTorch torchvision example:

```bash
python examples/python/compile_model.py \
  --frontend pytorch \
  --model resnet18 \
  --target-profile x86_64

python examples/python/run_model.py \
  --artifact-dir examples/artifacts/resnet18/x86_64 \
  --image tvm_cpp/images/cat.png
```

Supported starter PyTorch models are `resnet18`, `mobilenet_v2`, `squeezenet1_1`, and `shufflenet_v2_x1_0`. TensorFlow starter models are `mobilenet_v2` and `resnet50`.

ONNX and TFLite need a file path, input name, and input shape because those details are model-specific:

```bash
python examples/python/compile_model.py \
  --frontend onnx \
  --model-path path/to/model.onnx \
  --input-name input0 \
  --input-shape 1,3,224,224 \
  --target-profile x86_64
```

## Compile For Other Targets

Change only `--target-profile` when moving from local validation to another supported profile:

```bash
python examples/python/compile_model.py --frontend pytorch --model resnet18 --target-profile raspi4_aarch64
python examples/python/compile_model.py --frontend pytorch --model resnet18 --target-profile raspi_armv7
python examples/python/compile_model.py --frontend pytorch --model resnet18 --target-profile wasm32
python examples/python/compile_model.py --frontend pytorch --model resnet18 --target-profile c
```

Cross-compilation is only half the job. The target device also needs a compatible runtime program and compatible TVM runtime libraries. For embedded Linux targets, match the compiler and sysroot to the target OS image.

## Use The C++ Runtime Example

The C++ runner consumes the same artifact directory as the Python runtime. Build it after compiling a model:

```bash
cmake -S examples/cpp -B examples/cpp/build -G Ninja
cmake --build examples/cpp/build

examples/cpp/build/run_tvm_graph \
  --artifact-dir examples/artifacts/resnet18/x86_64 \
  --image tvm_cpp/images/cat.png
```

For cross-compiled artifacts, the C++ runner, compiled model library, TVM runtime library, and target OS ABI must all match.

## Where To Change Things

- Add or adjust model loaders in `examples/python/tvm_prep/model_zoo.py`.
- Add target profiles in `examples/python/tvm_prep/targets.py`.
- Change compile/export behavior in `examples/python/tvm_prep/compile.py`.
- Change Python runtime behavior in `examples/python/tvm_prep/runtime.py`.
- Change image preprocessing in `examples/python/tvm_prep/preprocess.py`.

Keep generated model outputs under `examples/artifacts/`. Do not commit generated `.so`, `.params`, `.json`, ONNX, or artifact directories.